# Teacher YOLO26m · 5 seed · Kaggle

Latih teacher deteksi supervised di Poltrada V3 untuk mengukur variasi hasil antar-seed.
Seed 42, 0, 2024, 7, dan 3407 mengikuti baseline pretrained N/S. Run All menjalankan
100 epoch per seed. Setiap seed memuat ulang bobot awal `yolo26m.pt` dan mengevaluasi
`best.pt` pada validation. Tambahkan dataset Poltrada V3 YOLO26 Clean dan secret WANDB_API_KEY di Kaggle.

Setiap seed mendapat folder dan ID W&B baru. Simpan referensi versi tetap dan SHA-256
dari `best_artifact` untuk menggunakan teacher baru pada KD. Artefak teacher `v50`
adalah rujukan untuk eksperimen KD yang sudah dikonfigurasi.

## 1. Konfigurasi eksperimen

In [ ]:
DATA = '/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml'
PROJECT = "/kaggle/working/results"

EXPERIMENT_ID = "02_teacher_yolo26m_multiseed"
METHOD = "none"
MODEL_WEIGHTS = "yolo26m.pt"  # Bisa diganti path file pretrained YOLO26m lokal.
SEEDS_TO_RUN = [42, 0, 2024, 7, 3407]
EPOCHS = 100
IMGSZ = 640
BATCH = 16
DEVICE = 0                   # GPU yang terlihat oleh kernel; "cpu" untuk uji lokal.
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
PATIENCE = 100
AMP = True
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = "val"
INSTALL_DEPENDENCIES = True
WANDB_ENABLED = True
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_ENTITY = "ilhamrafiqin-brawijaya-university"
WANDB_GROUP = "supervised"
BASE_WANDB_TAGS = ["teacher", "model-m", "poltrada-v3", EXPERIMENT_ID, 'kaggle']
TEACHER_CHECKPOINT = None

## 2. Preflight kernel dan W&B

In [ ]:
import gc
import hashlib
import importlib.metadata as metadata
import json
import os
import subprocess
import sys
import tempfile
import time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean, stdev

pins = {"ultralytics": "8.4.155", "wandb": "0.28.1"}


def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


preserved = {name: installed_version(name) for name in ("torch", "torchvision")}
if not all(preserved.values()):
    raise RuntimeError("Gunakan kernel yang sudah memiliki Torch dan torchvision.")
needed = [f"{name}=={version}" for name, version in pins.items() if installed_version(name) != version]
if needed:
    if not INSTALL_DEPENDENCIES:
        raise RuntimeError(f"Dependensi belum sesuai: {needed}")
    with tempfile.TemporaryDirectory(prefix="teacher-yolo26m-") as temp:
        constraints = Path(temp) / "torch-constraints.txt"
        constraints.write_text("".join(f"{name}=={version}\n" for name, version in preserved.items()), encoding="utf-8")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-c", str(constraints), *needed])
    if {name: installed_version(name) for name in preserved} != preserved:
        raise RuntimeError("Torch/torchvision berubah; gunakan kembali environment GPU awal.")

import torch
import ultralytics
import wandb
from ultralytics import YOLO, settings
from ultralytics.utils.torch_utils import init_seeds

if ultralytics.__version__ != pins["ultralytics"] or wandb.__version__ != pins["wandb"]:
    raise RuntimeError("Restart kernel setelah pemasangan dependensi.")
if DEVICE != "cpu" and not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA tidak tersedia pada kernel ini.")
settings.update({"wandb": False})
ENVIRONMENT_INFO = {"python": sys.version, "kernel": sys.executable, "torch": torch.__version__,
                    "torchvision": preserved["torchvision"], "cuda": torch.version.cuda,
                    "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE != "cpu" else None, **pins}
print(json.dumps(ENVIRONMENT_INFO, indent=2))
print("Dataset:", DATA, "| Output:", PROJECT)

if WANDB_ENABLED:
    key = os.environ.get("WANDB_API_KEY")
    if not key:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
    if not key:
        raise RuntimeError("Tambahkan WANDB_API_KEY melalui Kaggle Secrets.")
    wandb.login(key=key)
    del key

## 3. Periksa dataset train dan validation

In [ ]:
from ultralytics.data.utils import check_det_dataset


def sha256_file(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


if DATA != "coco8.yaml" and not Path(DATA).is_file():
    raise FileNotFoundError(f"Atur DATA ke data.yaml Poltrada V3 yang tersedia: {DATA}")
if EVAL_SPLIT not in {"val", "test"}:
    raise ValueError("Pilih EVAL_SPLIT val atau test.")
data_info = check_det_dataset(DATA, autodownload=DATA == "coco8.yaml", split=EVAL_SPLIT)
for split in {"train", "val", EVAL_SPLIT}:
    paths = data_info.get(split)
    if not paths or not all(Path(path).exists() for path in ([paths] if isinstance(paths, str) else paths)):
        raise FileNotFoundError(f"Split {split} tidak tersedia. Periksa path di YAML sumber {DATA}.")
ENVIRONMENT_INFO["dataset_yaml_sha256"] = sha256_file(data_info["yaml_file"])
ENVIRONMENT_INFO["class_names"] = [data_info["names"][index] for index in range(data_info["nc"])]
print(f"Split evaluasi: {EVAL_SPLIT} | Kelas: {ENVIRONMENT_INFO['class_names']}")

## 4. Training supervised dan antrean seed

In [ ]:
def log_status(stage, message):
    stamp = datetime.now().astimezone().strftime("%H:%M:%S")
    print(f"[{stamp}] [{stage}] {message}", flush=True)


KD_TRAINER = None
KD_TRAIN_ARGS = {}
KD_CONFIG = {}

def load_initial_model():
    global INITIAL_SHA256
    model = YOLO(MODEL_WEIGHTS)
    digest = sha256_file(model.ckpt_path)
    if INITIAL_SHA256 is not None and digest != INITIAL_SHA256:
        raise RuntimeError("Bobot pretrained berubah di tengah antrean seed.")
    INITIAL_SHA256 = digest
    return model


def summarize_seed_metrics(results, seeds):
    summary = {"seeds_requested": seeds, "seeds_completed": [item["seed"] for item in results],
               "initial_checkpoint_sha256": INITIAL_SHA256, "eval_split": EVAL_SPLIT, "metrics": {}}
    for name in ("map50_95", "map50", "precision", "recall"):
        values = [item[name] for item in results]
        summary["metrics"][name] = {"mean": mean(values), "sample_std": stdev(values) if len(values) > 1 else None}
    return summary

In [ ]:
@contextmanager
def exclusive_gpu():
    lock = Path(PROJECT).parent / ".yolo26m_teacher_gpu.lock"
    lock.parent.mkdir(parents=True, exist_ok=True)
    try:
        descriptor = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError as error:
        raise RuntimeError(f"GPU dikunci eksperimen lain: {lock}.") from error
    try:
        with os.fdopen(descriptor, "w") as stream:
            stream.write(json.dumps({"pid": os.getpid(), "experiment": EXPERIMENT_ID, "run": getattr(wb_run, "id", None)}))
        yield
    finally:
        lock.unlink(missing_ok=True)




def save_epoch_artifact(trainer):
    """Snapshot checkpoint setiap epoch; upload selesai sebelum last.pt ditimpa epoch berikutnya."""
    epoch = int(trainer.epoch) + 1
    last_path = Path(trainer.last)
    best_path = Path(trainer.best)
    if not last_path.is_file():
        raise RuntimeError(f"Checkpoint epoch {epoch} tidak ada: {last_path}")
    if trainer.batch_size != BATCH or trainer.args.imgsz != IMGSZ or bool(trainer.amp) != AMP:
        raise RuntimeError("Batch, resolusi, atau AMP efektif berubah dari konfigurasi.")
    wb_state["saved_epochs"].add(epoch)
    fitness = None if trainer.fitness is None else float(trainer.fitness)
    best_fitness = None if trainer.best_fitness is None else float(trainer.best_fitness)
    is_best = fitness is not None and best_fitness is not None and fitness == best_fitness
    if is_best:
        wb_state["best_epoch"] = epoch
    if wb_run is None:
        return
    best_digest = sha256_file(best_path) if is_best and best_path.is_file() else None
    artifact = wandb.Artifact(
        name=f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints",
        type="model",
        description="Checkpoint penuh per epoch: last.pt, best-to-date.pt, results.csv, args.yaml",
        metadata={
            "epoch": epoch, "fitness": fitness, "best_fitness": best_fitness,
            "is_best_epoch": bool(is_best), "best_epoch_so_far": wb_state["best_epoch"],
            "best_pt_sha256": best_digest,
            "experiment": EXPERIMENT_ID, "dataset": DATA, "seed": SEED,
        },
    )
    artifact.add_file(str(last_path), name="last.pt")
    if best_path.is_file():
        artifact.add_file(str(best_path), name="best.pt")
    for filename in ("results.csv", "args.yaml"):
        file_path = Path(trainer.save_dir) / filename
        if file_path.is_file():
            artifact.add_file(str(file_path), name=filename)
    aliases = [f"epoch-{epoch:03d}", "latest"]
    if is_best:
        aliases.append("best")
    # Synchronous wait menjamin isi artifact epoch ini selesai dikirim sebelum file lokal berubah.
    uploaded = wb_run.log_artifact(artifact, aliases=aliases)
    uploaded.wait()
    if best_digest:
        reference = uploaded.qualified_name
        version = reference.rpartition(":")[2]
        if not version.startswith("v") or not version[1:].isdigit():
            raise RuntimeError("Upload best.pt tidak mengembalikan referensi W&B berversi.")
        wb_state["best_artifact"] = {"reference": reference, "best_pt_sha256": best_digest}
    wb_state["uploaded_epochs"].add(epoch)
    print(f"W&B checkpoint: epoch={epoch}, best={is_best}, aliases={aliases}")

def log_epoch_metrics(trainer):
    """Satu record grafik per epoch (training loss, KD loss, validation, LR, best epoch)."""
    epoch = int(trainer.epoch) + 1
    # Ultralytics memanggil on_fit_epoch_end sekali lagi pada final_eval; abaikan panggilan ekstra.
    if epoch not in wb_state["uploaded_epochs"] or epoch in wb_state["logged_epochs"]:
        return
    loss = trainer.label_loss_items(trainer.tloss, prefix="train")
    payload = {"epoch": epoch, **{k: float(v) for k, v in loss.items()},
               **{k: float(v) for k, v in trainer.lr.items()}}
    for key, value in trainer.metrics.items():
        try:
            payload[key] = float(value)
        except (ValueError, TypeError):
            continue
    payload["checkpoint/is_best_epoch"] = int(wb_state["best_epoch"] == epoch)
    payload["checkpoint/best_epoch_so_far"] = wb_state["best_epoch"] or 0
    wb_run.log(payload, step=epoch)
    wb_state["logged_epochs"].add(epoch)

def run_full_training():
    model = None
    with exclusive_gpu():
        try:
            init_seeds(SEED, deterministic=True)
            if DEVICE != "cpu":
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats(DEVICE)
            model = load_initial_model()
            train_args = dict(
                data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE,
                weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
                project=str(SESSION_DIR), name="training", exist_ok=False,
                workers=WORKERS, patience=PATIENCE, plots=True,
                amp=AMP, pretrained=True, save=True,
            )
            train_args.update(KD_TRAIN_ARGS)
            if wb_run is not None:
                wb_run.config.update({"training_args": train_args, "initial_checkpoint_sha256": INITIAL_SHA256})
            model.add_callback("on_model_save", bind_seed_callback(save_epoch_artifact))
            model.add_callback("on_fit_epoch_end", bind_seed_callback(log_epoch_metrics))
            start = time.perf_counter()
            model.train(**train_args)
            directory = Path(model.trainer.save_dir)
            best = directory / "weights/best.pt"
            if not best.is_file():
                raise FileNotFoundError(f"Checkpoint best.pt tidak ditemukan: {best}")
            if len(wb_state["saved_epochs"]) != EPOCHS:
                raise RuntimeError("Training berakhir sebelum anggaran epoch lengkap.")
            return {"run_dir": str(directory), "best_pt": str(best),
                    "last_pt": str(directory / "weights/last.pt"),
                    "training_seconds": time.perf_counter() - start,
                    "completed_epochs": len(wb_state["saved_epochs"]), "best_epoch": wb_state["best_epoch"],
                    "initial_checkpoint_sha256": INITIAL_SHA256, "best_artifact": wb_state.get("best_artifact")}
        finally:
            model = None
            gc.collect()
            if DEVICE != "cpu":
                torch.cuda.empty_cache()


def validate_seed_list(seeds):
    """Reject ambiguous queues before opening any run or creating a model."""
    if not isinstance(seeds, (list, tuple)) or not seeds:
        raise ValueError("SEEDS_TO_RUN harus berupa daftar seed yang tidak kosong.")
    if any(isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed < 2**32 for seed in seeds):
        raise ValueError("Setiap seed harus integer dalam rentang 0 sampai 2**32-1.")
    if len(seeds) != len(set(seeds)):
        raise ValueError("SEEDS_TO_RUN tidak boleh memiliki seed duplikat.")
    return list(seeds)


def seed_run_name(seed):
    return f"base_teacher_yolo26m_seed{seed}"


def bind_seed_callback(callback):
    """Bind the active seed/run/state; reject stale callbacks instead of mixing logs."""
    seed, run, state = SEED, wb_run, wb_state
    def bound(trainer):
        if SEED != seed or wb_run is not run or wb_state is not state:
            raise RuntimeError("Callback berasal dari seed/run lain; logging dihentikan.")
        return callback(trainer)
    return bound


def start_seed_run():
    global wb_run
    NEW_WANDB_RUN_ID = wandb.util.generate_id()
    log_status("W&B", f"Membuat run baru: name={WANDB_RUN_NAME}, id={NEW_WANDB_RUN_ID}, resume=never...")
    diagnostic = DATA == "coco8.yaml" or DEVICE == "cpu"
    wb_run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=WANDB_RUN_NAME,
        id=NEW_WANDB_RUN_ID, resume="never", settings=wandb.Settings(init_timeout=90),
        group=f"diagnostic-{WANDB_GROUP}" if diagnostic else WANDB_GROUP, job_type="training",
        tags=[*WANDB_TAGS, "diagnostic" if diagnostic else "research"],
        config={"experiment": EXPERIMENT_ID, "method": METHOD, "model": MODEL_WEIGHTS,
                "dataset": DATA, "eval_split": EVAL_SPLIT, "seed": SEED,
                "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "device": str(DEVICE),
                "workers": WORKERS, "optimizer": OPTIMIZER, "learning_rate": LR,
                "patience": PATIENCE, "amp": AMP, "weight_decay": WEIGHT_DECAY,
                "warmup_epochs": WARMUP_EPOCHS, "close_mosaic": CLOSE_MOSAIC,
                "environment": ENVIRONMENT_INFO, "seeds_to_run": list(SEEDS_TO_RUN)})
    log_status("W&B", f"Run aktif: {wb_run.url}")


def run_one_seed(seed, queue_dir):
    """Complete one independent run, including evaluation/upload/finish, before returning."""
    global SEED, SESSION_DIR, WANDB_RUN_NAME, WANDB_TAGS, wb_run, wb_state
    if wandb.run is not None:
        raise RuntimeError("Masih ada run W&B aktif; seed baru tidak boleh dimulai.")
    SEED = seed
    WANDB_RUN_NAME = seed_run_name(seed) + ("_smoke" if DATA == "coco8.yaml" or DEVICE == "cpu" else "")
    WANDB_TAGS = [*BASE_WANDB_TAGS, f"seed{seed}"]
    SESSION_DIR = Path(queue_dir) / f"seed-{seed}"
    SESSION_DIR.mkdir(parents=True, exist_ok=False)
    wb_run = None
    wb_state = {"saved_epochs": set(), "uploaded_epochs": set(), "logged_epochs": set(),
                "best_epoch": None, "epoch_upload_seconds": []}
    try:
        init_seeds(seed, deterministic=True)
        if WANDB_ENABLED:
            start_seed_run()
        log_status("SEED", f"Memulai seed {seed}: {EPOCHS} epoch, teacher baru dari {MODEL_WEIGHTS}.")
        training = run_full_training()
        (SESSION_DIR / "training_result.json").write_text(json.dumps(training, indent=2), encoding="utf-8")
        with exclusive_gpu():
            result = evaluate_final_training(training)
        result.update(seed=seed, wandb_run_id=getattr(wb_run, "id", None), wandb_run_name=WANDB_RUN_NAME,
                      wandb_url=getattr(wb_run, "url", None), seed_directory=str(SESSION_DIR))
        (SESSION_DIR / "seed_result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
        # finish() waits for the run's remaining W&B sync; the next seed starts afterwards.
        if wb_run is not None:
            wb_run.finish()
        log_status("SEED", f"Seed {seed} selesai.")
        return result
    except BaseException:
        if wb_run is not None:
            try:
                wb_run.finish(exit_code=1)
            except Exception as finish_error:
                log_status("W&B", f"Penutupan run gagal ({type(finish_error).__name__}); antrean dihentikan.")
        raise
    finally:
        wb_run = None
        gc.collect()
        if DEVICE != "cpu":
            torch.cuda.empty_cache()


def run_seed_queue():
    """Serial queue: persist completed seeds and stop immediately on any failure."""
    seeds = validate_seed_list(SEEDS_TO_RUN)
    global INITIAL_SHA256
    INITIAL_SHA256 = None
    if wandb.run is not None:
        raise RuntimeError("Selesaikan run W&B aktif sebelum menjalankan antrean ini.")
    token = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    queue_dir = Path(PROJECT) / EXPERIMENT_ID / f"multiseed_{token}"
    queue_dir.mkdir(parents=True, exist_ok=False)
    results = []
    progress = {"experiment": EXPERIMENT_ID, "seeds_to_run": seeds, "completed": [],
                "failed_seed": None, "status": "running"}
    progress_path = queue_dir / "multiseed_progress.json"
    progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
    log_status("QUEUE", f"Seed serial {seeds}; hasil: {queue_dir}.")
    for seed in seeds:
        try:
            result = run_one_seed(seed, queue_dir)
            results.append(result)
            (queue_dir / "multiseed_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
            (queue_dir / "multiseed_summary.json").write_text(
                json.dumps(summarize_seed_metrics(results, seeds), indent=2, allow_nan=False), encoding="utf-8")
            progress["completed"].append({"seed": seed, "wandb_run_id": result["wandb_run_id"],
                                          "seed_directory": result["seed_directory"]})
            progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
        except BaseException:
            progress.update(failed_seed=seed, status="failed")
            progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
            log_status("QUEUE", f"Berhenti pada seed {seed}. Hasil seed selesai tetap tersimpan; untuk lanjut isi SEEDS_TO_RUN dengan seed yang belum selesai.")
            raise
    progress["status"] = "completed"
    progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
    log_status("QUEUE", f"Selesai: {len(results)} run terpisah.")
    return results

## 5. Evaluasi best checkpoint dan ekspor per seed

In [ ]:
def evaluate_final_training(training):
    best_model = metrics = None
    try:
        run_dir = Path(training["run_dir"])
        best_pt = Path(training["best_pt"])
        training_seconds = training["training_seconds"]
        # Evaluasi dari checkpoint terbaik (bukan last.pt).
        best_model = YOLO(str(best_pt))
        metrics = best_model.val(
            data=DATA, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH,
            device=DEVICE, plots=True, verbose=False,
            project=str(SESSION_DIR), name="evaluation", exist_ok=False,
        )

        from ultralytics.utils.torch_utils import get_flops
        parameters = sum(p.numel() for p in best_model.model.parameters())
        try:
            gflops = float(get_flops(best_model.model, imgsz=IMGSZ))
        except (AttributeError, ImportError, RuntimeError, TypeError, ValueError) as error:
            print("GFLOPs tidak tersedia:", error)
            gflops = None

        results = {
            "experiment": EXPERIMENT_ID,
            "method": METHOD,
            "model": MODEL_WEIGHTS,
            "dataset": DATA,
            "eval_split": EVAL_SPLIT,
            "seed": SEED,
            "epochs": EPOCHS,
            "imgsz": IMGSZ,
            "batch": BATCH,
            "optimizer": OPTIMIZER,
            "learning_rate": LR,
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "map50": float(metrics.box.map50),
            "map50_95": float(metrics.box.map),
            "parameters": int(parameters),
            "gflops": gflops,
            "inference_ms_per_image": float(metrics.speed.get("inference", 0.0)),
            "training_seconds": float(training_seconds),
            "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "teacher_checkpoint": str(TEACHER_CHECKPOINT) if METHOD != "none" else None,
            "best_pt": str(best_pt),
        }
        if DEVICE != "cpu":
            results["peak_gpu_memory_gb"] = float(torch.cuda.max_memory_allocated() / 1024**3)
        print(json.dumps(results, indent=2, ensure_ascii=False))
        # File lokal tetap tersimpan oleh Ultralytics, W&B menyimpan versi artifact tiap epoch.
        run_dir.mkdir(parents=True, exist_ok=True)
        results.update(best_epoch=training["best_epoch"], completed_epochs=training["completed_epochs"],
                       initial_checkpoint_sha256=training["initial_checkpoint_sha256"],
                       best_pt_sha256=sha256_file(best_pt), best_artifact=training["best_artifact"])
        (run_dir / "metrics.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
        configuration = {
            "experiment": EXPERIMENT_ID, "dataset": DATA, "evaluation_split": EVAL_SPLIT,
            "model": MODEL_WEIGHTS, "teacher": TEACHER_CHECKPOINT if METHOD != "none" else None,
            "seed": SEED, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
            "optimizer": OPTIMIZER, "lr0": LR, "device": DEVICE,
            "workers": WORKERS, "patience": PATIENCE, "amp": AMP,
            "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
            "close_mosaic": CLOSE_MOSAIC,
            "kd_hyperparameters": KD_CONFIG,
            "wandb_enabled": WANDB_ENABLED, "wandb_project": WANDB_PROJECT, "environment": ENVIRONMENT_INFO,
        }
        (run_dir / "config.json").write_text(json.dumps(configuration, indent=2, ensure_ascii=False), encoding="utf-8")

        if wb_run is not None:
            summary = {f"final/{key}": val for key, val in results.items()
                       if isinstance(val, (int, float)) and not isinstance(val, bool) and val is not None}
            wb_run.log(summary)
            wb_run.summary.update(summary)
            wb_run.summary["best_epoch"] = wb_state["best_epoch"]
            wb_run.summary["best_artifact"] = results["best_artifact"]
            wb_run.summary["checkpoint_collection"] = f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints"
            wb_run.summary["uploaded_checkpoint_epochs"] = len(wb_state["uploaded_epochs"])

            # Ringkasan/config dan grafik sebagai Artifact terpisah, tanpa menggandakan checkpoint model.
            final_artifact = wandb.Artifact(
                name=f"{EXPERIMENT_ID}-{wb_run.id}-results", type="results",
                metadata={"experiment": EXPERIMENT_ID, "seed": SEED, "best_epoch": wb_state["best_epoch"]},
            )
            for filename in ("metrics.json", "config.json", "results.csv", "args.yaml"):
                candidate = run_dir / filename
                if candidate.is_file():
                    final_artifact.add_file(str(candidate), name=filename)
            for folder_name, folder in (("training", run_dir),
                                        ("evaluation", Path(getattr(metrics, "save_dir", run_dir)))):
                if folder.exists():
                    for plot_path in sorted(folder.glob("*.png")):
                        key = f"plots/{folder_name}/{plot_path.stem}"
                        wb_run.log({key: wandb.Image(str(plot_path))})
                        final_artifact.add_file(str(plot_path), name=f"{folder_name}/{plot_path.name}")
            wb_run.log_artifact(final_artifact, aliases=["final"]).wait()
            print("W&B:", wb_run.url)
            print("Checkpoint alias: epoch-001 ... epoch-%03d, latest, best" % len(wb_state["uploaded_epochs"]))

        print("Hasil lokal:", run_dir)
        print("  - weights/best.pt, weights/last.pt")
        print("  - results.csv, args.yaml")
        print("  - metrics.json, config.json")

        return results

    finally:
        best_model = metrics = None
        gc.collect()
        if DEVICE != "cpu":
            torch.cuda.empty_cache()

## 6. Jalankan antrean

Satu seed menuntaskan training, evaluasi, upload, dan penutupan W&B sebelum seed berikutnya.
Ringkasan `multiseed_summary.json` memuat mean dan simpangan baku sampel per metrik.
Bila antrean berhenti, periksa `multiseed_progress.json` dan isi `SEEDS_TO_RUN` dengan seed
yang masih perlu dijalankan. Pilih satu platform/GPU untuk seluruh lima seed.

In [ ]:
MULTISEED_RESULTS = run_seed_queue()
print(json.dumps(MULTISEED_RESULTS, indent=2))